# 06 · Comparação dos modelos

Junta as predições salvas pelos notebooks 03–05 no teste do RePro: F1 com intervalo de confiança, comparação pareada contra o melhor modelo e McNemar.

In [ ]:
# Configuração: no Colab, clona o repositório, instala o pacote `absa`
# e guarda cache do LLM, modelos e predições no Drive (sobrevive à queda da sessão)
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e "."
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models /content/drive/MyDrive/pln-absa-olist/interim
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/interim data/interim

In [ ]:
import pandas as pd
from absa.config import INTERIM, LABEL_COLUMNS
from absa.metrics import bootstrap_ci, macro_f1, mcnemar, paired_bootstrap, per_aspect_report, tuple_f1
from absa.repro import load_repro
treino, val, teste, _ = load_repro()
print(len(treino), len(val), len(teste))

preds = {}
for arq in sorted(INTERIM.glob("pred_repro_teste_*.csv")):
    p = pd.read_csv(arq).set_index("review_id").loc[teste.review_id, LABEL_COLUMNS].reset_index(drop=True)
    preds[arq.stem.removeprefix("pred_repro_teste_")] = p
list(preds)

In [ ]:
linhas = []
for nome, p in preds.items():
    f1, lo, hi = bootstrap_ci(teste, p)
    linhas.append({"modelo": nome, "f1_macro": f1, "ic_inf": lo, "ic_sup": hi, "f1_tupla": tuple_f1(teste, p)})
tabela = pd.DataFrame(linhas).sort_values("f1_macro", ascending=False).round(3)
tabela

## O melhor modelo é significativamente melhor que os outros?

In [ ]:
melhor = tabela.modelo.iloc[0]
comp = []
for nome in tabela.modelo.iloc[1:]:
    pb = paired_bootstrap(teste, preds[melhor], preds[nome])
    mc = mcnemar(teste, preds[melhor], preds[nome])
    comp.append({"contra": nome, "diferenca": pb["diferenca"], "ic95": tuple(round(x, 3) for x in pb["ic95"]),
                 "p_bootstrap": pb["p_valor"], "p_mcnemar": mc["p_valor"]})
print("melhor:", melhor)
pd.DataFrame(comp).round(4)

## Humano × LLM como fonte de rótulo

In [ ]:
for enc in ["tfidf_logreg", "bertimbau", "albertina"]:
    h, l = preds.get(f"{enc}_humano"), preds.get(f"{enc}_llm")
    if h is not None and l is not None:
        pb = paired_bootstrap(teste, h, l)
        print(f"{enc}: humano − LLM = {pb['diferenca']:+.3f} (IC95% {pb['ic95'][0]:+.3f} a {pb['ic95'][1]:+.3f})")

## Por aspecto (melhor modelo)

In [ ]:
per_aspect_report(teste, preds[melhor]).round(3)

## Mudança de domínio: amostra anotada da Olist (opcional)

In [ ]:
from absa.config import ANNOTATED
if (ANNOTATED / "olist_anotado.csv").exists():
    from absa.sampling import load_olist_gold
    print("Olist anotada disponível: rode o melhor modelo nela no notebook 07")
else:
    print("olist_anotado.csv ainda não existe")